<a href="https://colab.research.google.com/github/richardjong22/ISYS2001-Richard/blob/main/student_budget_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Budget & Savings Goal Helper
## Six-Step Evidence Documentation

### Step 1: Understand the Problem
University students often struggle to keep track of their monthly spending and lack clarity on whether their savings goals are achievable based on their current transaction history. The goal of this assistant is to process a student's transaction history (CSV format), evaluate their current net monthly savings, compare it against a target savings goal over a specific timeframe, and provide actionable financial advice.

### Step 2: Identify Inputs and Outputs
- **Inputs**:
  - `csv_file`: A CSV file containing transaction history with columns: `Date`, `Description`, `Category`, `Amount` (positive for income, negative for expenses).
  - `target_amount` (float): Desired total savings goal (e.g., $600.00).
  - `target_months` (int): Number of months to achieve the goal (e.g., 3 months).
  - `user_question` (str, optional): Specific question for the AI financial coach.
- **Outputs**:
  - `summary_text` (str): Calculated breakdown of total income, expenses, current net savings, required monthly savings, and goal status (On Track vs. Shortfall).
  - `ai_advice` (str): Personal financial tips generated by Google Gemini API based on the transaction summary.

### Step 3: Work an Example by Hand
- **Sample Data**:
  - Income: Salary (+$1,500.00)
  - Expenses: Rent (-$800.00), Groceries (-$200.00), Dining (-$100.00)
- **Calculations**:
  - Total Income = $1,500.00
  - Total Expenses = $800.00 + $200.00 + $100.00 = $1,100.00
  - Current Monthly Net Savings = $1,500.00 - $1,100.00 = $400.00
  - Target: $600.00 in 3 months
  - Required Monthly Savings = $600.00 / 3 = $200.00 / month
  - Monthly Gap = $400.00 - $200.00 = +$200.00 (Surplus)
- **Status**: On Track! Exceeds target by $200.00/month.

### Step 4: Write Pseudocode
FUNCTION calculate_savings_plan(csv_file, target_amount, target_months):
    IF target_amount <= 0 OR target_months <= 0 THEN
        RETURN "Error: Target amount and timeframe must be greater than zero."
    END IF
    
    READ csv_file INTO dataframe
    VERIFY required columns ('Date', 'Description', 'Category', 'Amount') exist
    
    CALCULATE total_income = SUM of Amount WHERE Amount > 0
    CALCULATE total_expenses = ABSOLUTE SUM of Amount WHERE Amount < 0
    CALCULATE current_monthly_savings = total_income - total_expenses
    
    CALCULATE required_monthly_savings = target_amount / target_months
    CALCULATE monthly_gap = current_monthly_savings - required_monthly_savings
    
    IF monthly_gap >= 0 THEN
        SET status = "ON TRACK"
    ELSE
        SET status = "SHORTFALL DETECTED"
    END IF
    
    FORMAT and RETURN financial summary string
END FUNCTION

In [ ]:
import pandas as pd

def calculate_savings_plan(csv_file, target_amount: float, target_months: int) -> str:
    """
    Analyzes transaction history from a CSV file and calculates savings goal feasibility.
    """
    if target_amount <= 0 or target_months <= 0:
        return "Error: Target amount and timeframe must be greater than zero."

    if csv_file is None:
        return "Error: Please upload a valid CSV file."

    try:
        if isinstance(csv_file, str):
            df = pd.read_csv(csv_file)
        else:
            df = pd.read_csv(csv_file.name)

        required_columns = {'Date', 'Description', 'Category', 'Amount'}
        if not required_columns.issubset(df.columns):
            return f"Error: CSV missing required columns. Must contain: {required_columns}"

        total_income = df[df['Amount'] > 0]['Amount'].sum()
        total_expenses = abs(df[df['Amount'] < 0]['Amount'].sum())

        current_monthly_savings = total_income - total_expenses
        required_monthly_savings = target_amount / target_months
        monthly_gap = current_monthly_savings - required_monthly_savings

        if monthly_gap >= 0:
            status = f"✅ ON TRACK! You exceed your required monthly target by ${monthly_gap:,.2f}."
        else:
            status = f"⚠️ SHORTFALL DETECTED! You need to reduce spending by ${abs(monthly_gap):,.2f}/month."

        summary = (
            f"=== FINANCIAL ANALYSIS SUMMARY ===\n"
            f"• Total Income: ${total_income:,.2f}\n"
            f"• Total Expenses: ${total_expenses:,.2f}\n"
            f"• Current Monthly Net Savings: ${current_monthly_savings:,.2f}\n\n"
            f"=== SAVINGS GOAL PLAN ===\n"
            f"• Target Goal: ${target_amount:,.2f} in {target_months} month(s)\n"
            f"• Required Monthly Savings: ${required_monthly_savings:,.2f}/month\n"
            f"• Goal Status: {status}\n"
        )
        return summary

    except Exception as e:
        return f"Error processing CSV file: {str(e)}"

In [ ]:
import io

def test_calculate_savings_plan():
    # Test Case 1: Edge Case - Target bernilai negatif/nol
    res_err = calculate_savings_plan("dummy.csv", -100, 3)
    assert "Error: Target amount" in res_err

    # Test Case 2: Valid CSV Data
    csv_data = """Date,Description,Category,Amount
2026-09-01,Salary,Income,1500.00
2026-09-05,Rent,Housing,-800.00
2026-09-10,Groceries,Food,-200.00
"""
    fake_csv = io.StringIO(csv_data)
    df_test = pd.read_csv(fake_csv)
    df_test.to_csv("test_sample.csv", index=False)

    res_success = calculate_savings_plan("test_sample.csv", 600, 3)
    assert "ON TRACK" in res_success
    assert "Total Income: $1,500.00" in res_success

    print("✅ All unit tests passed successfully!")

test_calculate_savings_plan()

✅ All unit tests passed successfully!


In [ ]:
# Install library Google GenAI resmi
!pip install -q google-genai

from google.genai import client as genai_client
from google.colab import userdata
import gradio as gr

# Setup Client
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    ai = genai_client.Client(api_key=api_key)
    print("✅ Gemini API Client successfully configured!")
except Exception as e:
    print(f"⚠️ Error loading GOOGLE_API_KEY: {str(e)}")

def ask_gemini_coach(summary_text: str, user_question: str) -> str:
    prompt = f"""
    You are Financial Sage, a friendly AI financial advisor for university students.
    Based on this financial analysis:

    {summary_text}

    User Question: {user_question}

    Provide concise, practical, and encouraging financial advice in 3-4 bullet points.
    """
    try:
        # Menggunakan versi model terbaru yang diminta oleh Google API
        response = ai.models.generate_content(
            model='gemini-3.6-flash',
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"Gemini API Error: {str(e)}"

def process_app(csv_file, target_amount, target_months, user_question):
    summary = calculate_savings_plan(csv_file, target_amount, target_months)

    if "Error:" in summary:
        return summary, "N/A - Please resolve the input error above."

    q = user_question if user_question else "How can I improve my overall savings rate?"
    advice = ask_gemini_coach(summary, q)
    return summary, advice

# Gradio Interface Setup
demo = gr.Interface(
    fn=process_app,
    inputs=[
        gr.File(label="Upload Transactions CSV"),
        gr.Number(label="Target Savings Goal ($)", value=600),
        gr.Number(label="Target Timeframe (Months)", value=3),
        gr.Textbox(label="Ask Financial Sage (Optional)", placeholder="e.g., How can I reduce my weekly spending?")
    ],
    outputs=[
        gr.Textbox(label="Financial Analysis Summary", lines=8),
        gr.Textbox(label="AI Financial Coach Advice", lines=8)
    ],
    title="Student Budget & Savings Goal Helper 🎓💰",
    description="Upload your transaction history in CSV format and set a target savings goal to receive personalized financial analysis and AI-driven coaching."
)

demo.launch(share=True)

✅ Gemini API Client successfully configured!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9cdaddd2d6dc64207a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
